# TFG Experiments

Cuaderno principal de exploracion del TFG, orientado a barridos con `DFM`, comprobacion de backbones y visualizacion de ejemplos del dataset reconstruido.


## Objetivo del cuaderno

- reunir el bloque mas cercano al flujo experimental del TFG;
- dejar una base para barridos sobre `MVTec-AD` con `DFM`;
- documentar exploracion de backbones sin descargas innecesarias;
- visualizar ejemplos del dataset propio y de inferencia.


## Como leer este notebook

Este cuaderno se parece mas a una libreta de trabajo principal que a una guia paso a paso. La reorganizacion actual intenta convertirlo en algo mas interpretable: primero se muestra el contexto del dataset, luego la inspeccion de backbones y por ultimo la plantilla del barrido experimental y la inferencia.


## Entorno y rutas


In [ ]:
from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DATA_DIR = ROOT / "data"
DOCS_DIR = ROOT / "docs"
ANOMALIB_DIR = ROOT / "anomalib"
RESULTS_DIR = ROOT / "results"
NOTES_DIR = ROOT / "notes"
NOTEBOOKS_DIR = ROOT / "notebooks"
NOTES_DIR.mkdir(exist_ok=True)

RAW_DATASET_DIR = DATA_DIR / "mandarins_pynq_raw"
CROPPED_DATASET_DIR = DATA_DIR / "mandarins_pynq_cropped"
AUGMENTED_DATASET_DIR = DATA_DIR / "mandarins_pynq_augmented"
INFERENCE_NORMAL_IMAGE = DATA_DIR / "inference_normal.png"
INFERENCE_ANOMALY_IMAGE = DATA_DIR / "inference_anomaly.png"

try:
    from anomalib.config import get_configurable_parameters as _anomalib_probe
    ANOMALIB_AVAILABLE = True
except Exception:
    ANOMALIB_AVAILABLE = False

TIMM_AVAILABLE = importlib.util.find_spec("timm") is not None
CV2_AVAILABLE = importlib.util.find_spec("cv2") is not None
KERAS_AVAILABLE = importlib.util.find_spec("keras") is not None

environment_summary = pd.DataFrame(
    [
        {"item": "repo_root", "value": str(ROOT)},
        {"item": "anomalib_available", "value": ANOMALIB_AVAILABLE},
        {"item": "timm_available", "value": TIMM_AVAILABLE},
        {"item": "cv2_available", "value": CV2_AVAILABLE},
        {"item": "keras_available", "value": KERAS_AVAILABLE},
        {"item": "raw_dataset_exists", "value": RAW_DATASET_DIR.exists()},
        {"item": "augmented_dataset_exists", "value": AUGMENTED_DATASET_DIR.exists()},
        {"item": "results_exists", "value": RESULTS_DIR.exists()},
    ]
)
environment_summary


## Resumen del dataset y del caso de uso


In [ ]:
def dataset_count_table(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for label in ["normal", "abnormal"]:
        label_dir = dataset_dir / label
        count = len([path for path in sorted(label_dir.iterdir()) if path.is_file()]) if label_dir.exists() else 0
        rows.append({"dataset": dataset_dir.name, "label": label, "images": count})
    return pd.DataFrame(rows)


def summarize_mandarin_datasets() -> pd.DataFrame:
    frames = []
    for dataset_dir in [RAW_DATASET_DIR, CROPPED_DATASET_DIR, AUGMENTED_DATASET_DIR]:
        if dataset_dir.exists():
            frames.append(dataset_count_table(dataset_dir))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["dataset", "label", "images"])


def show_image_grid(image_paths, titles, figsize=(12, 4)) -> None:
    fig, axes = plt.subplots(1, len(image_paths), figsize=figsize)
    if len(image_paths) == 1:
        axes = [axes]
    for axis, image_path, title in zip(axes, image_paths, titles):
        axis.imshow(Image.open(image_path))
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()

tfg_dataset_summary = summarize_mandarin_datasets()
tfg_dataset_summary


La memoria del TFG insiste en que el caso propio no es un benchmark grande, sino un escenario pequeno y realista. Por eso tiene sentido abrir el cuaderno mostrando el tamano de cada particion y algunos ejemplos visuales del conjunto de trabajo.


In [ ]:
example_paths = [
    next(iter(sorted((RAW_DATASET_DIR / "normal").iterdir()))),
    next(iter(sorted((RAW_DATASET_DIR / "abnormal").iterdir()))),
    INFERENCE_NORMAL_IMAGE,
    INFERENCE_ANOMALY_IMAGE,
]
show_image_grid(example_paths, ["Raw normal", "Raw abnormal", "Inference normal", "Inference anomaly"], figsize=(14, 4))


## Exploracion rapida de backbones


Antes de lanzar configuraciones sobre `DFM`, conviene inspeccionar que backbones y que capas son razonables para extraer representaciones. Esta parte no pretende agotar la exploracion, solo dejar visibles algunos candidatos que tienen sentido dentro del repo reconstruido.


In [ ]:
if not TIMM_AVAILABLE:
    print("timm no esta disponible.")
else:
    import timm

    def inspect_timm_backbone(model_name: str):
        model = timm.create_model(model_name, pretrained=False)
        return [name for name, _ in model.named_children()]

    pd.DataFrame({"layer": inspect_timm_backbone("efficientnet_l2")[:12]})


In [ ]:
import torchvision
resnet101 = torchvision.models.resnet101(weights=None)
pd.DataFrame({"layer": [name for name, _ in resnet101.named_children()]})


## Plantilla de barrido con DFM sobre MVTec


Este bloque representa el nucleo experimental del cuaderno: una rejilla sencilla de categorias, `score_type` y `pooling_kernel_size`. Se deja como plantilla para que el barrido sea comprensible y modificable sin disparar automaticamente horas de entrenamiento en cada ejecucion.


In [ ]:
if not ANOMALIB_AVAILABLE:
    print("anomalib no esta disponible.")
else:
    from pytorch_lightning import Trainer
    from anomalib.config import get_configurable_parameters
    from anomalib.data.mvtec import MVTec
    from anomalib.models.dfm.lightning_model import Dfm
    
    def build_dfm_experiment(score_type: str, pooling_kernel_size: int, category: str):
        config_path = ANOMALIB_DIR / "models" / "dfm" / "custom.yaml"
        config = get_configurable_parameters(config_path=str(config_path))
        datamodule = MVTec(
            root=str(DATA_DIR / "mvtec_anomaly_detection"),
            category=category,
            image_size=256,
            train_batch_size=32,
            test_batch_size=32,
            num_workers=8,
            task="segmentation",
            seed=42,
        )
        datamodule.setup()
        model = Dfm(backbone="resnet18", layer="layer3", score_type=score_type, pooling_kernel_size=pooling_kernel_size)
        callbacks = []
        return datamodule, model, callbacks, config

    def grid_definition(categories, score_types=("fre", "nll"), pooling_sizes=(2, 4, 6)) -> pd.DataFrame:
        rows = []
        for category in categories:
            for score_type in score_types:
                for pooling_size in pooling_sizes:
                    rows.append({
                        "category": category,
                        "score_type": score_type,
                        "pooling_kernel_size": pooling_size,
                    })
        return pd.DataFrame(rows)

    categories = ["tile", "toothbrush", "transistor", "wood", "zipper"]
    grid_definition(categories)


## Plantilla de inferencia con PaDiM


La inferencia aparece aqui como bloque final porque sirve para cerrar el flujo experimental: despues de estudiar configuraciones y barridos, la pregunta practica es como aplicar un checkpoint a una imagen concreta y visualizar la respuesta del modelo.


In [ ]:
if not ANOMALIB_AVAILABLE:
    print("Se omite la plantilla de inferencia porque anomalib no esta disponible.")
else:
    from torch.utils.data import DataLoader
    from pytorch_lightning import Trainer
    from anomalib.data.inference import InferenceDataset
    from anomalib.models.padim.lightning_model import PadimLightning
    from anomalib.pre_processing.transforms import Denormalize

    def run_padim_inference(image_path: Path, checkpoint_path: Path):
        model = PadimLightning.load_from_checkpoint(str(checkpoint_path))
        trainer = Trainer()
        dataloader = DataLoader(InferenceDataset(str(image_path), image_size=(256, 256)))
        return trainer.predict(model=model, dataloaders=dataloader)[0]

    def plot_prediction(output) -> None:
        image = Denormalize()(output["image"][0])
        anomaly_map = output["anomaly_maps"][0].cpu().numpy().squeeze()
        pred_mask = output["pred_masks"][0].cpu().numpy().squeeze()
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(image)
        axes[0].set_title("Image")
        axes[1].imshow(anomaly_map)
        axes[1].set_title("Anomaly map")
        axes[2].imshow(pred_mask)
        axes[2].set_title("Predicted mask")
        for axis in axes:
            axis.axis("off")
        plt.tight_layout()
        plt.show()

    print("Funciones de inferencia listas para ejecucion manual con checkpoints existentes.")


## Conclusiones del cuaderno

Este notebook queda como el punto mas cercano a la narrativa experimental del TFG. Sirve para entender que se prueba, con que parametros se podria repetir el barrido y como se conectan los resultados cuantitativos con ejemplos de inferencia sobre imagenes concretas.
